<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_7_variant_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ai_ball_7.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RScD6nl_1_teIc-eGZ_Tao-A5BP6xo1J

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A `Council_Brain` that combines several independently-trained brains by majority vote
5. A **continuous relaxation** training method (`Ball_Trap_Brain.train_continuous`) that trains a brain with gradient descent instead of the genetic algorithm, and can optionally keep the trained continuous parameters usable directly (`predict_continuous`, `evaluate_continuous`) instead of only ever discretizing them back into the original discrete brain
6. An optional, guarded cell that reproduces the original end-to-end council training run

## Imports & tokenizer

In [ ]:

import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus a gradient-based continuous relaxation trainer, split across a few methods:
- `train_continuous(...)`: trains the continuous parameters with gradient descent (see its docstring for details on the relaxation scheme). Temperature cools on a fixed linear schedule from `temp_start` to `temp_end` over `epochs`. Returns `(loss_history, accuracy_history)`.
- `predict_continuous(input_str, ...)`: answer a question using the **continuous** brain directly, without discretizing anything -- no dependency on `deflectors_list` etc at all.
- `evaluate_continuous(test_cases, ...)`: like `evaluate_performance`, but scores `predict_continuous` instead of `compute_input`.
- `commit_continuous()`: discretize already-trained continuous parameters into the discrete brain without training any further.

**Two pollution bugs, now fixed, that affected discrete *and* continuous output alike:**
1. *Leading-position leakage.* The loss only scored a right-aligned tail of each trajectory; leading positions outside that window had no training signal at all and could leak arbitrary symbols into the real output. Fixed by folding "should this position emit at all" into the *same* per-position loss as a genuine "emit nothing" class -- `never_fired = 1 - out_dist.sum()` already forms a valid (K+1)-way distribution with the K real symbols, so every position (aligned or not) now has a well-defined target under one unified objective, instead of a second, separately-weighted penalty term that destabilized training by fighting the main loss over shared deflectors.
2. *Repeat/self-loop cycling.* Once a trajectory hit a hole, the "first hole reached" objective stopped caring what happened for the rest of its `cap` steps -- so a hole deflector could cycle with another hole (or even itself) and just re-emit symbols for the whole `cap` budget (e.g. `"3333333333"`, `"2323232323"`). Fixed with `repeat_penalty_weight` (default 0.3), which penalizes the expected number of *additional* hole-hits after the first one per trajectory -- computed from the same per-example forward pass as the main loss, so it targets whatever cycle each specific trajectory is actually at risk of.

**`commit` flag:** `train_continuous(..., commit=True)` (the default) discretizes the trained parameters back into this instance's own `deflectors_list`, `alphabet_directions` and `alphabet_start_positions` -- so every other method (`compute_input`, `ricochet`, `clone`, `Mutate`, ...) keeps working exactly as before, just with gradient-trained values. Pass `commit=False` to leave the discrete brain completely untouched and only keep the trained continuous version around for `predict_continuous`/`evaluate_continuous` -- useful for comparing continuous vs. discrete behavior, or for iterating with more `train_continuous` calls before ever committing to a discrete brain (call `commit_continuous()` whenever you're ready to).

**How `token_acc` is scored (printed during training):** within each epoch's batch, for every trajectory position (aligned *or* blank), it checks whether the predicted class -- top symbol if emit-probability >= 0.5, else "blank" -- matches the target class, and reports the fraction that do. It's read off the same numbers already used for the loss -- no extra simulation -- but it's still a *character-position* metric, not a *question* metric: it doesn't check whether a whole answer matches end to end, and it's computed on the soft relaxation, not the real discrete brain. For real, whole-question exact-match accuracy, use `evaluate_performance` (discrete) or `evaluate_continuous` (continuous).

In [ ]:

class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        directions = [random.choice([1,-1]) for _ in range(alphabet)]
        return directions

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        directions.pop(index)
        position += sum(directions)
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output, self.holes_multiplier)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = 1 if self.alphabet_directions[i] == -1 else -1

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

    def train_continuous(self, test_cases, epochs=300, lr=0.05, batch_size=16,
                          temp_start=2.5, temp_end=0.5, sharpness=0.75,
                          repeat_penalty_weight=0.3,
                          warm_start=True, commit=True, verbose=True, reset=None):
        """
        Continuous relaxation training.

        Every discrete choice the brain makes is replaced with a differentiable
        stand-in for the duration of training:
          - each deflector's destination index      -> a softmax over all deflectors
          - each deflector's hole/bounce identity    -> a sigmoid (hole probability)
          - each deflector's output symbol (if hole) -> a softmax over self.output
          - each deflector's bounce value (if not)   -> a free real-valued scalar
          - each alphabet direction (+-1)            -> tanh of a free scalar
          - each alphabet start position              -> a softmax over all deflectors

        The ricochet loop is unrolled for `cap` steps and the hard indexing /
        branching / while-loop of the discrete simulator is replaced by
        probability-weighted ("expected value") versions of the same
        arithmetic, so the whole forward pass is differentiable and can be
        trained with gradient descent against the task loss instead of the
        genetic algorithm's mutate-and-select loop. A temperature is annealed
        from `temp_start` down to `temp_end` over training so the softmax
        distributions sharpen towards one-hot (i.e. towards the original
        discrete model) as training progresses.

        **Epochs vs. steps.** Each epoch is a full pass over a shuffled copy of
        `test_cases`, split into batches of `batch_size`; the optimizer takes one
        gradient step per batch (so an epoch is several steps, not one). Every
        example is seen exactly once per epoch, rather than each iteration
        drawing an independent random sample of the data (which gives no
        per-epoch coverage guarantee and can revisit or skip examples). The
        temperature schedule and the returned histories are still indexed by
        epoch (one entry per epoch, averaged across that epoch's batches/steps).

        Because the discrete model can emit a variable number of output
        symbols per input, this relaxation makes one simplifying assumption:
        it treats the *first hole reached* by each character's trajectory as
        that character's contribution to the output, and aligns those
        per-character predictions against the target string position by
        position, right-aligned (the last trajectory position lines up with
        the last target character, and so on).

        Every trajectory position is trained, not just the aligned ones.
        `never_fired = 1 - out_dist.sum()` (the probability a trajectory ends
        the simulation without ever emitting anything) turns each position's
        prediction into a genuine (K+1)-way distribution: the K real output
        symbols, plus one "emit nothing" class, all summing to 1 by
        construction. Leading input positions that fall outside the aligned
        window (i.e. that have no corresponding target character at all) are
        given "emit nothing" as their target class in that same distribution.
        Earlier versions of this method left those leading positions with no
        training signal whatsoever, so they were free to fire and leak
        arbitrary symbols into the real, full output regardless of how low
        the loss on the aligned positions got; a later attempt fixed that
        with a second, separately-weighted penalty term, but that fought the
        main loss for control of shared deflectors and could destabilize
        training. Folding "should this position emit at all" into the *same*
        per-position cross-entropy as the main task avoids both problems --
        there's one loss, and every position (aligned or not) has a
        well-defined target under it.

        There's a related, narrower pathology this doesn't cover: once a
        trajectory reaches a hole, this objective stops caring what happens
        for the rest of that trajectory's `cap` steps. In the real, discrete
        brain, if that trajectory keeps landing on holes afterward (a direct
        self-loop, a 2-cycle between two holes, or any longer cycle -- all
        equally invisible to the "first hole" objective above), `ricochet`
        just keeps re-appending symbols for the rest of the `cap` steps (e.g.
        `"3333333333"`, `"2323232323"`). `repeat_penalty_weight` (default 0.3)
        penalizes the expected number of *additional* hole-hits after the
        first one for every trajectory in the batch (`extra_fire` from
        `_c_simulate`) -- computed from the same per-example forward pass
        already used for the main loss, so it directly targets whatever
        cycle each specific trajectory is actually at risk of, rather than
        only guarding against direct self-loops.

        The trained continuous parameters are kept on this instance (usable
        via `predict_continuous` / `evaluate_continuous`, see below) whether
        or not you discretize. If `commit=True` (the default), every
        continuous parameter is *also* discretized (argmax / sign / rounding)
        and written directly back into this instance's own `deflectors_list`,
        `alphabet_directions` and `alphabet_start_positions` attributes, so
        every other method on this class (compute_input, ricochet, clone,
        Mutate, display_deflectors, ...) keeps working exactly as before,
        just with gradient-trained values in place of the old ones. Pass
        `commit=False` to leave those discrete attributes untouched and only
        keep the trained continuous version (e.g. if you want to compare
        continuous vs. discrete behavior, or keep iterating with
        `train_continuous` again before ever committing to a discrete brain).

        `reset` controls whether the continuous parameters are (re)initialized
        from the current discrete brain before this call trains them further.
        Default (`None`) resets only if this instance has no continuous
        parameters yet (i.e. this is the first `train_continuous` call);
        subsequent calls on the same instance continue training whatever
        continuous parameters already exist instead of throwing them away and
        starting over from the discrete brain again. Pass `reset=True` to force
        a fresh re-initialization (e.g. after changing `warm_start`), or
        `reset=False` to require continuation and raise if there's nothing to
        continue.

        Returns `(loss_history, accuracy_history)`, both per-epoch lists of
        floats (one entry per full pass over the data, averaged across that
        epoch's batches). `accuracy_history` is a cheap "token accuracy" over
        *every* trajectory position seen during the epoch (aligned or blank):
        it checks whether that position's predicted class -- its top symbol if
        its emit-probability is >= 0.5, else "blank" -- matches the target
        class, and reports the fraction that do. It's computed for free from
        the same numbers already used for the loss -- no extra simulation is
        run -- so while it now also reflects whether leading positions are
        correctly suppressed, it's still not the same as full-sequence
        exact-match correctness on the real, discrete brain (see
        `evaluate_continuous` / `evaluate_performance` for that).
        """
        import torch

        if reset is None:
            reset = not hasattr(self, '_c_meta')
        elif reset is False and not hasattr(self, '_c_meta'):
            raise RuntimeError(
                "reset=False but this brain has no continuous parameters yet -- "
                "call train_continuous(...) at least once (or pass reset=True) first."
            )
        if reset:
            self._init_continuous_params(warm_start=warm_start, sharpness=sharpness)
        c = self._c_meta
        output_index = c['output_index']

        optimizer = torch.optim.Adam(
            [self._c_dest_logits, self._c_hole_logit, self._c_symbol_logits,
             self._c_bounce_val, self._c_direction_param, self._c_start_logits], lr=lr
        )

        history = []
        acc_history = []
        n_batches = max(1, (len(test_cases) + batch_size - 1) // batch_size)
        temp = temp_start

        # Tokenization doesn't depend on parameters or temperature, so it only
        # needs to happen once, not once per epoch. Examples that tokenize to
        # no known symbols are dropped up front instead of being re-detected
        # (and re-tokenized) on every single epoch.
        prepared = []
        for inp, target in test_cases:
            tokens = TokenizeInput(inp, self.alphabet)
            indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
            if not indices:
                continue
            target_indices = [output_index[ch] for ch in target if ch in output_index]
            prepared.append((indices, target_indices))

        for epoch in range(epochs):
            frac = epoch / max(1, epochs - 1)
            temp = temp_start * (temp_end / temp_start) ** frac

            shuffled = list(prepared)
            random.shuffle(shuffled)

            epoch_loss_sum = 0.0
            epoch_count = 0
            epoch_correct = 0

            for b in range(n_batches):
                batch = shuffled[b * batch_size:(b + 1) * batch_size]
                if not batch:
                    continue

                optimizer.zero_grad()

                # The deflector-level distributions (dest_probs, hole_p,
                # symbol_probs, weighted_abs, directions_soft, expected start
                # positions) depend only on the current parameters and temp --
                # not on which examples are in the batch -- so they're computed
                # ONCE per optimizer step here and shared across every example's
                # trajectories, instead of being recomputed from scratch inside
                # a per-example call (which is what made batch_size have almost
                # no effect on speed before: batch_size=1 vs. 16 did the same
                # D x D softmax work per example either way).
                shared = self._c_precompute(temp)

                # All trajectories (every character of every example in this
                # batch) are simulated together in one vectorized pass rather
                # than looping per example in Python -- see _c_simulate_batch.
                indices_list = [indices for indices, _ in batch]
                sim_results = self._c_simulate_batch(indices_list, temp, shared=shared)

                total_loss = torch.tensor(0.0)
                count = 0
                correct = 0
                repeat_loss = torch.tensor(0.0)
                repeat_count = 0
                for (indices, target_indices), (outs, never_fired, extra_fire) in zip(batch, sim_results):
                    L = len(outs)
                    n = min(L, len(target_indices))
                    # Right-align: the *last* n trajectory positions correspond to the *last* n
                    # target characters. Test inputs typically share a literal, invariant prefix
                    # (e.g. every "1+i" starts with the same "1+"), so left-aligning would
                    # repeatedly score that shared, input-invariant trajectory against different
                    # targets -- gradients then can't distinguish examples at all. The trailing
                    # characters of the input are where the example-specific information (e.g.
                    # the digits of i) actually lives, so align from the end instead.
                    out_offset = L - n
                    tgt_offset = len(target_indices) - n
                    # Score EVERY position, not just the aligned tail. `out_dist[k]` and
                    # `never_fired` already form one consistent (K+1)-way distribution over
                    # "emit symbol k" / "emit nothing" (never_fired = 1 - out_dist.sum() by
                    # construction), so positions before out_offset -- which have no target
                    # character at all -- are simply given "emit nothing" as their target class
                    # in that SAME distribution, rather than being left with no training signal
                    # (which is what let them leak arbitrary symbols into the real output) or
                    # fought over with a separately-weighted penalty term (which destabilized
                    # training by competing against the main loss on shared deflectors).
                    for i in range(L):
                        if i < out_offset:
                            target_prob = never_fired[i]
                            target_class = c['K']   # the "blank" class
                        else:
                            target_idx = target_indices[tgt_offset + (i - out_offset)]
                            target_prob = outs[i][target_idx]
                            target_class = target_idx
                        total_loss = total_loss - torch.log(target_prob + 1e-8)

                        p_emit = float(outs[i].sum().item())
                        pred_class = int(torch.argmax(outs[i]).item()) if p_emit >= 0.5 else c['K']
                        if pred_class == target_class:
                            correct += 1
                        count += 1

                        repeat_loss = repeat_loss + extra_fire[i]
                        repeat_count += 1

                if count == 0:
                    continue
                task_loss = total_loss / count
                combined_loss = task_loss
                if repeat_penalty_weight and repeat_count:
                    combined_loss = combined_loss + repeat_penalty_weight * (repeat_loss / repeat_count)
                combined_loss.backward()
                optimizer.step()

                epoch_loss_sum += task_loss.item() * count
                epoch_count += count
                epoch_correct += correct

            if epoch_count == 0:
                continue
            avg_loss = epoch_loss_sum / epoch_count
            avg_acc = epoch_correct / epoch_count
            history.append(avg_loss)
            acc_history.append(avg_acc)

            if verbose and (epoch % max(1, epochs // 20) == 0 or epoch == epochs - 1):
                print(f"[continuous] epoch {epoch}: loss={avg_loss:.4f} token_acc={avg_acc:.3f} "
                      f"temp={temp:.2f} batches={n_batches}")

        # Remember the temperature training ended at, so predict_continuous /
        # evaluate_continuous default to something close to what was actually trained.
        self._c_meta['last_temp'] = temp

        if commit:
            self._discretize_continuous_into(self)

        return history, acc_history

    def _init_continuous_params(self, warm_start=True, sharpness=0.75):
        """Build (or rebuild) this instance's continuous relaxation parameters from its
        current discrete state, storing them on the instance so they persist beyond a
        single train_continuous call and can be used by predict_continuous /
        evaluate_continuous / _discretize_continuous_into."""
        import torch

        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        output_index = {sym: k for k, sym in enumerate(self.output)}
        positions = torch.arange(D, dtype=torch.float32)
        abs_diff = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # abs_diff[d, d'] = |d - d'|

        CONF = 4.0  # logit magnitude used so a warm-started softmax closely matches the current discrete choice
        dest_logits = torch.zeros(D, D)
        hole_logit = torch.zeros(D)
        symbol_logits = torch.zeros(D, K)
        bounce_val = torch.zeros(D)
        direction_param = torch.zeros(A)
        start_logits = torch.zeros(A, D)

        if warm_start:
            for i, (dest, val) in enumerate(self.deflectors_list):
                dest_logits[i, dest] = CONF
                if isinstance(val, int):
                    hole_logit[i] = -CONF
                    bounce_val[i] = float(val)
                else:
                    hole_logit[i] = CONF
                    symbol_logits[i, output_index[val]] = CONF
            for i, d in enumerate(self.alphabet_directions):
                direction_param[i] = CONF if d == 1 else -CONF
            for i, p in enumerate(self.alphabet_start_positions):
                start_logits[i, p] = CONF
        else:
            dest_logits.normal_(0, 0.1)
            hole_logit.normal_(0, 0.1)
            symbol_logits.normal_(0, 0.1)
            bounce_val.normal_(0, 1.0)
            direction_param.normal_(0, 0.1)
            start_logits.normal_(0, 0.1)

        for p in (dest_logits, hole_logit, symbol_logits, bounce_val, direction_param, start_logits):
            p.requires_grad_(True)

        self._c_dest_logits = dest_logits
        self._c_hole_logit = hole_logit
        self._c_symbol_logits = symbol_logits
        self._c_bounce_val = bounce_val
        self._c_direction_param = direction_param
        self._c_start_logits = start_logits
        self._c_meta = dict(D=D, A=A, K=K, cap=10, ball_energy=float(self.ball_energy),
                             output_index=output_index, positions=positions, abs_diff=abs_diff,
                             sharpness=sharpness, last_temp=None)

    def _c_soft_index_dist_batched(self, travel_values, temp):
        """Batched version of the soft-position-to-deflector distribution.
        `travel_values` is a 1-D tensor of shape (N,) (one travel value per
        trajectory); returns an (N, D) tensor of per-trajectory distributions
        over deflectors."""
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        diff = c['positions'].unsqueeze(0) - travel_values.unsqueeze(1)  # (N, D)
        diff = torch.remainder(diff + c['D'] / 2, c['D']) - c['D'] / 2
        logits = -(diff ** 2) / (c['sharpness'] * temp)
        return F.softmax(logits, dim=1)

    def _c_precompute(self, temp):
        """Compute the deflector-level distributions (dest/hole/symbol
        probabilities, expected hop distance, soft direction, and expected
        start position) from the current continuous parameters and `temp`.

        These depend only on the parameters and `temp` -- not on which
        examples are being trained on -- so callers should compute this ONCE
        per gradient step and share it across every example/trajectory in
        that step's batch, rather than recomputing it per example (which is
        what the original per-example `_c_simulate` calls did)."""
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        dest_probs = F.softmax(self._c_dest_logits / temp, dim=1)          # (D, D)
        hole_p = torch.sigmoid(self._c_hole_logit)                         # (D,)
        symbol_probs = F.softmax(self._c_symbol_logits / temp, dim=1)      # (D, K)
        weighted_abs = (dest_probs * c['abs_diff']).sum(dim=1)             # (D,)
        bounce_term = (1 - hole_p) * self._c_bounce_val                    # (D,)
        directions_soft = torch.tanh(self._c_direction_param)              # (A,)
        start_probs_all = F.softmax(self._c_start_logits / temp, dim=1)    # (A, D)
        expected_start_all = (start_probs_all * c['positions']).sum(dim=1)  # (A,)
        return dict(dest_probs=dest_probs, hole_p=hole_p, symbol_probs=symbol_probs,
                    weighted_abs=weighted_abs, bounce_term=bounce_term,
                    directions_soft=directions_soft, expected_start_all=expected_start_all)

    def _c_simulate_batch(self, batch_indices, temp, shared=None):
        """Vectorized version of the differentiable soft ricochet simulation:
        runs every trajectory (every character, of every example in the
        batch) through the `cap`-step unroll TOGETHER as one batched tensor
        computation, instead of looping per character in Python with scalar
        tensor ops (which is what made training slow -- lots of tiny ops with
        real per-op Python/dispatch overhead relative to their actual compute
        cost).

        `batch_indices` is a list of lists of alphabet indices, one list per
        example. `shared` is the dict from `_c_precompute(temp)`; if omitted
        it's computed here (once, not once per example).

        Returns a list of `(outputs, never_fired, extra_fire)` tuples, one per
        example, in the same format `_c_simulate` used to return for a single
        example -- see its docstring (still available below) for the meaning
        of each."""
        import torch

        c = self._c_meta
        if shared is None:
            shared = self._c_precompute(temp)
        dest_probs = shared['dest_probs']
        hole_p = shared['hole_p']
        symbol_probs = shared['symbol_probs']
        weighted_abs = shared['weighted_abs']
        bounce_term = shared['bounce_term']
        directions_soft = shared['directions_soft']
        expected_start_all = shared['expected_start_all']

        example_lengths = [len(indices) for indices in batch_indices]
        flat_travel = []
        for indices in batch_indices:
            if not indices:
                continue
            idx_t = torch.tensor(indices, dtype=torch.long)
            dirs = directions_soft[idx_t]                 # (L,)
            starts = expected_start_all[idx_t]             # (L,)
            total_dir = dirs.sum()
            travel = starts + (total_dir - dirs)           # (L,) matches Brain.travel()
            flat_travel.append(travel)

        if not flat_travel:
            return [([], [], []) for _ in batch_indices]

        travel_cat = torch.cat(flat_travel, dim=0)          # (N,)
        N = travel_cat.shape[0]

        state = self._c_soft_index_dist_batched(travel_cat, temp)   # (N, D)
        be = torch.full((N,), float(c['ball_energy']))
        alive = torch.ones(N)          # soft "be > 0 and cap_idx < cap" gate
        not_hole_yet = torch.ones(N)   # soft "no hole reached yet" gate
        out_dist = torch.zeros(N, c['K'])
        extra = torch.zeros(N)         # expected re-fires after the first hole

        for _ in range(c['cap']):
            p_hole = state @ hole_p                              # (N,)
            p_bounce_energy = state @ bounce_term                # (N,)
            exp_dist = state @ weighted_abs                      # (N,)

            already_fired = alive * (1.0 - not_hole_yet)
            extra = extra + already_fired * p_hole

            p_first_hole = alive * not_hole_yet * p_hole                          # (N,)
            # sym_dist[n, k] = sum_d state[n,d] * hole_p[d] * symbol_probs[d,k] / p_hole[n]
            sym_dist = (state * hole_p.unsqueeze(0)) @ symbol_probs               # (N, K)
            # Clamp the denominator (rather than just adding a tiny epsilon) so
            # that when p_hole is near zero for a trajectory, the division
            # doesn't blow up into a large, noisy gradient -- the result is
            # already being multiplied by p_first_hole (also ~0 in that case),
            # so clamping barely affects the forward value but keeps backward
            # well-behaved.
            sym_dist = sym_dist / p_hole.clamp(min=1e-3).unsqueeze(1)
            out_dist = out_dist + p_first_hole.unsqueeze(1) * sym_dist

            energy_delta = p_hole + p_bounce_energy - exp_dist
            be = be + energy_delta
            alive = alive * torch.sigmoid(be * 2.0)
            not_hole_yet = not_hole_yet * (1 - p_hole)

            state = state @ dest_probs                            # (N, D)

        never_fired = 1.0 - out_dist.sum(dim=1)                   # (N,)

        results = []
        ptr = 0
        for L in example_lengths:
            if L == 0:
                results.append(([], [], []))
                continue
            outs_e = [out_dist[ptr + i] for i in range(L)]
            nf_e = [never_fired[ptr + i] for i in range(L)]
            extra_e = [extra[ptr + i] for i in range(L)]
            results.append((outs_e, nf_e, extra_e))
            ptr += L
        return results

    def _c_simulate(self, input_indices, temp):
        """Single-example convenience wrapper around `_c_simulate_batch` (kept
        for backward compatibility / external callers). Runs the
        differentiable, soft ricochet simulation using this instance's
        persisted continuous parameters (self._c_*). Returns (outputs,
        never_fired, extra_fire), one entry per input character: `outputs[i]`
        is a probability distribution over self.output (via the "first hole
        reached" convention -- see train_continuous's docstring), `never_fired[i]`
        is the probability that trajectory never emits anything within `cap`
        steps, and `extra_fire[i]` is the expected number of *additional*
        hole-hits after the first one (i.e. how much the trajectory keeps
        re-firing instead of settling down for the rest of its cap steps).

        For training on more than one example, prefer calling
        `_c_simulate_batch` directly with all examples' indices at once (and a
        `shared = self._c_precompute(temp)` reused across the batch) -- looping
        this single-example method per example recomputes the shared
        deflector-level distributions from scratch every time, which is much
        slower."""
        return self._c_simulate_batch([input_indices], temp)[0]

    def _discretize_continuous_into(self, target_brain):
        """Argmax/sign/round this instance's continuous parameters (self._c_*) and write
        them into target_brain's discrete attributes (deflectors_list, alphabet_directions,
        alphabet_start_positions)."""
        import torch
        c = self._c_meta
        with torch.no_grad():
            for i in range(c['D']):
                dest = int(torch.argmax(self._c_dest_logits[i]).item())
                if torch.sigmoid(self._c_hole_logit[i]).item() > 0.5:
                    val = self.output[int(torch.argmax(self._c_symbol_logits[i]).item())]
                else:
                    val = int(round(self._c_bounce_val[i].item()))
                target_brain.deflectors_list[i][0] = dest
                target_brain.deflectors_list[i][1] = val
            for i in range(c['A']):
                target_brain.alphabet_directions[i] = 1 if self._c_direction_param[i].item() > 0 else -1
                target_brain.alphabet_start_positions[i] = int(torch.argmax(self._c_start_logits[i]).item())

    def commit_continuous(self):
        """Discretize this instance's already-trained continuous parameters (from a prior
        train_continuous(..., commit=False) call) into its discrete deflectors_list /
        alphabet_directions / alphabet_start_positions, without training any further."""
        if not hasattr(self, '_c_meta'):
            raise RuntimeError("No continuous parameters to commit -- call train_continuous(...) first.")
        self._discretize_continuous_into(self)

    def predict_continuous(self, input_str, temp=None, return_probs=False):
        """
        Answer `input_str` using the *continuous* (un-discretized) brain directly --
        no argmax/rounding of parameters, no dependency on deflectors_list/
        alphabet_directions/alphabet_start_positions at all. Requires a prior
        train_continuous(...) call (with or without commit=True) on this instance.

        Unlike training (which only scores a target-aligned tail of each trajectory),
        this decodes *every* input character's trajectory: if its "never fire" probability
        is below 0.5 it's treated as emitting, and the most likely symbol from its output
        distribution is appended, in the same left-to-right order compute_input uses.

        `temp` defaults to the temperature training ended at (colder = closer to the
        discretized/deterministic behavior; pass a higher temp to see softer, more
        exploratory behavior). Set `return_probs=True` to also get back, for every input
        character, `(probability it emits at all, {symbol: probability})`.
        """
        import torch

        if not hasattr(self, '_c_meta'):
            raise RuntimeError(
                "No continuous parameters on this brain yet -- call train_continuous(...) "
                "first (pass commit=False if you don't want it to also overwrite the "
                "discrete deflectors_list/alphabet_directions/alphabet_start_positions)."
            )
        c = self._c_meta
        if temp is None:
            temp = c['last_temp'] if c['last_temp'] is not None else 0.5

        tokens = TokenizeInput(input_str, self.alphabet)
        indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
        if not indices:
            return ("", []) if return_probs else ""

        with torch.no_grad():
            outs, never_fired, _extra_fire = self._c_simulate(indices, temp)

        decoded = []
        probs = []
        for out_dist, nf in zip(outs, never_fired):
            p_emit = 1.0 - nf.item()
            sym_probs = {sym: p for sym, p in zip(self.output, out_dist.tolist())}
            probs.append((p_emit, sym_probs))
            if p_emit >= 0.5:
                decoded.append(self.output[int(torch.argmax(out_dist).item())])

        result = "".join(decoded)
        return (result, probs) if return_probs else result

    def evaluate_continuous(self, test_cases, temp=None, ratio=False):
        """Like evaluate_performance, but scores predict_continuous (the un-discretized
        brain) against test_cases instead of compute_input (the discretized/discrete
        brain). Returns the average score (0/1 exact-match per case, or a 0-1 similarity
        ratio if ratio=True)."""
        total = 0.0
        for inp, answer in test_cases:
            pred = self.predict_continuous(inp, temp=temp)
            if ratio:
                total += difflib.SequenceMatcher(None, pred, answer).ratio()
            else:
                total += 1 if pred == answer else 0
        return total / len(test_cases)

## Test case generation

In [ ]:

import random
import string
def generate_test_cases(amount, number, indiv=True):

  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:
    test_cases = [(f'1+{i}', f'{1+i}') for i in range(amount)]

  return test_cases

## Fitness evaluation

Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.

In [ ]:

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

## Genetic algorithm (`main`)

Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.

In [ ]:

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

# Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation


alphabet = [f'{i}' for i in range(10)] + ['+'] #+ list("abcdefghijklmnopqrstuvwxyz")  # Define the alphabet used in the brains
output = [f'{i}' for i in range(10)]

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

training_1 = generate_test_cases(amount=100, number='1', indiv=True)

## `Council_Brain` & council training

In [ ]:

from collections import Counter

class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately.
        The council votes on the final answer by finding the most common output.
        """
        votes = []
        for brain in self.brains:
            # Each brain outputs a list of characters; join them to form the string
            output_str = brain.predict_continuous(input_val)
            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

def train_council(num_brains, training_data, alphabet, output_format, generations=50, pop_size=100):
    """
    Trains a council where each brain is trained separately using the genetic algorithm.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(100, 10, alphabet, output, 2)
        # Initialize a fresh brain for this specific council member
        #base_brain = Ball_Trap_Brain(100, 10, alphabet, output_format, 2)

        # Train separately using the existing main GA loop
        for j in range(1):
          print(f'---{j}---')
          base_brain.train_continuous(
              training_data,
              epochs=generations,       # the batched/vectorized simulate makes far fewer epochs necessary
              lr=0.1,
              batch_size=50,     # >1 now actually helps: shared deflector distributions are computed
                                  # once per step and every trajectory in the batch is simulated together
              warm_start=True,
              commit=False,   # keep the discrete brain untouched; only train the continuous version
              verbose=True,
              temp_start=2.5,
              temp_end=0.5   # anneal close to one-hot so commit_continuous's argmax matches what was trained
          )
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

## Continuous relaxation training — demo

Trains a brain with gradient descent (requires `torch`; installed via `pip install torch` if missing) with `commit=False`, so the discrete brain is left completely untouched, and answers questions directly with the continuous version via `predict_continuous`. Then calls `commit_continuous()` and compares against the real, discrete `compute_input` -- with both pollution bugs fixed, these should now largely agree with each other, with no more repeated-character garbage.

In [ ]:

# # !pip install torch --quiet   # uncomment if torch isn't already installed

# import random
# random.seed(0)

# demo_brain = Ball_Trap_Brain(50, 10, alphabet, output, 2)
# cases = generate_test_cases(amount=100, number='1', indiv=False)  # '1+i' -> str(1+i)
# share = 0.5
# demo_cases = cases[:int(share*len(cases))]
# demo_test = cases[int(share*len(cases)):]
# loss_history, acc_history = demo_brain.train_continuous(
#     demo_cases,
#     epochs=100,       # the batched/vectorized simulate makes far fewer epochs necessary
#     lr=0.1,
#     batch_size=250,     # >1 now actually helps: shared deflector distributions are computed
#                         # once per step and every trajectory in the batch is simulated together
#     warm_start=True,
#     commit=False,   # keep the discrete brain untouched; only train the continuous version
#     verbose=True,
#     temp_start=2.5,
#     temp_end=0.5   # anneal close to one-hot so commit_continuous's argmax matches what was trained
# )
# print('token_acc[0] ->', acc_history[0], ' token_acc[-1] ->', acc_history[-1])

# print()
# print('=== CONTINUOUS predictions ===')
# for inp, ans in demo_cases[:10]:
#     pred = demo_brain.predict_continuous(inp)
#     print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
# cont_exact = demo_brain.evaluate_continuous(demo_cases, ratio=False)
# cont_ratio = demo_brain.evaluate_continuous(demo_cases, ratio=True)
# print('continuous exact-match:', cont_exact, ' ratio:', cont_ratio)
# print('continuous test match:', demo_brain.evaluate_continuous(demo_test, ratio=False))

# # demo_brain.commit_continuous()
# # print()
# # print('=== DISCRETE predictions (after commit_continuous) ===')
# # for inp, ans in demo_cases[:10]:
# #     pred = ''.join(demo_brain.compute_input(inp))
# #     print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
# # disc_exact = sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_cases) / len(demo_cases)
# # disc_ratio = sum(evaluate_performance(i, a, demo_brain, ratio=True) for i, a in demo_cases) / len(demo_cases)
# # print('discrete exact-match:', disc_exact, ' ratio:', disc_ratio)

## Full council training run (optional, slow)

This reproduces the original script's `if __name__ == '__main__':` block: it trains a 50-brain council with the genetic algorithm (400 generations each) and evaluates it. This can take a long time, so it's guarded behind `RUN_FULL_TRAINING`.

In [ ]:

RUN_FULL_TRAINING = True  # set to True to actually run the full council training

if RUN_FULL_TRAINING:
    # Generate the test cases using the existing generation function
    test_cases = generate_test_cases(amount=100, number='1', indiv=False)
    #random.shuffle(test_cases)

    # Split the dataset
    training_0 = test_cases[:int(0.5 * len(test_cases))]
    training_test = test_cases[int(0.5 * len(test_cases)):]


    # Define council parameters
    COUNCIL_SIZE = 50      # Number of voting brains in the council
    GENERATIONS = 200     # Generations per brain
    POP_SIZE = 100        # Population size per generation

    # Train the council (each brain is trained entirely separately)
    print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
    council = train_council(
        num_brains=COUNCIL_SIZE,
        training_data=training_0,
        alphabet=alphabet,
        output_format=output,
        generations=GENERATIONS,
        pop_size=POP_SIZE
    )



    # Evaluate the entire council's performance against the testing set
    # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
    print("\nEvaluating Council performance on test batch...")
    council_fitness, _ = evaluate_population_parallel([council], training_test, False)
    council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
    print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")

Initializing Council Training with 50 brains...

 Training Council Brain 1 / 50
---0---
[continuous] epoch 0: loss=1.9216 token_acc=0.321 temp=2.50 batches=1
[continuous] epoch 10: loss=1.6463 token_acc=0.405 temp=2.31 batches=1
[continuous] epoch 20: loss=1.4579 token_acc=0.511 temp=2.13 batches=1
[continuous] epoch 30: loss=1.2653 token_acc=0.516 temp=1.96 batches=1
[continuous] epoch 40: loss=1.0109 token_acc=0.653 temp=1.81 batches=1
[continuous] epoch 50: loss=0.8498 token_acc=0.705 temp=1.67 batches=1
[continuous] epoch 60: loss=0.7410 token_acc=0.695 temp=1.54 batches=1
[continuous] epoch 70: loss=0.6644 token_acc=0.721 temp=1.42 batches=1
[continuous] epoch 80: loss=0.6284 token_acc=0.721 temp=1.31 batches=1
[continuous] epoch 90: loss=0.6022 token_acc=0.721 temp=1.21 batches=1
[continuous] epoch 100: loss=0.5865 token_acc=0.742 temp=1.11 batches=1
[continuous] epoch 110: loss=0.5743 token_acc=0.721 temp=1.03 batches=1
[continuous] epoch 120: loss=0.5630 token_acc=0.768 temp=0.